### Python Notebook used for Figure 7.2 in the Master thesis "Shadow Removal and Mesh Improvements for Photogrammetry-based 3D Modelling of Built Environments".

In [ ]:
from __future__ import annotations

import gc
import os
import sys
import time
import tracemalloc
from pathlib import Path

import cv2
import geopandas as gpd
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
import torch
from scipy.ndimage import label as scipy_label
from skimage.segmentation import mark_boundaries

sys.path.insert(0, "..")
from fetch import build_camera, build_item_url, fetch_item

print("Torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())


In [ ]:
# Timing / memory context manager 

class Benchmark:
    """Context manager that records wall-clock time and peak RSS (MB)."""
    def __enter__(self):
        tracemalloc.start()
        self._t0 = time.perf_counter()
        return self

    def __exit__(self, *_):
        self.elapsed = time.perf_counter() - self._t0
        _, peak = tracemalloc.get_traced_memory()
        self.peak_mb = peak / 1e6
        tracemalloc.stop()

    def __repr__(self):
        return f"{self.elapsed:.2f}s  peak {self.peak_mb:.0f} MB"

# Visualisation helpers

def show(*imgs, titles=None, figsize_per=(5, 5), cmap=None, bgr=True):
    """Display one or more images side-by-side."""
    n = len(imgs)
    fig, axes = plt.subplots(1, n, figsize=(figsize_per[0]*n, figsize_per[1]))
    if n == 1:
        axes = [axes]
    for ax, img, title in zip(axes, imgs, titles or [""]*n):
        if img.ndim == 3 and bgr:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        ax.imshow(img, cmap=cmap)
        ax.set_title(title, fontsize=9)
        ax.axis("off")
    plt.tight_layout()
    plt.show()


def overlay_masks(img_bgr, masks: dict[str, np.ndarray],
                  colours: dict[str, tuple] | None = None,
                  alpha: float = 0.45) -> np.ndarray:
    """Blend a dict of binary masks (name→bool array) onto an BGR image."""
    default_palette = [
        (255, 165,   0),  # orange
        (0,   200,   0),  # green
        (0,     0, 255),  # red (BGR)
        (255,   0, 255),  # magenta
        (0,   255, 255),  # yellow
        (128, 128, 128),  # grey
        (255, 255,   0),  # cyan
    ]
    overlay = img_bgr.copy()
    for i, (name, mask) in enumerate(masks.items()):
        colour = (colours or {}).get(name, default_palette[i % len(default_palette)])
        coloured = np.full_like(img_bgr, colour)
        overlay[mask > 0] = (
            alpha * coloured[mask > 0] + (1 - alpha) * overlay[mask > 0]
        ).astype(np.uint8)
    return overlay


def segment_overlay(img_bgr, segments: np.ndarray, alpha: float = 0.6) -> np.ndarray:
    """Colour each segment a random hue and show boundaries."""
    rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB).astype(float) / 255
    coloured = np.zeros_like(rgb)
    rng = np.random.default_rng(42)
    for sid in np.unique(segments):
        if sid == 0:
            continue
        colour = rng.uniform(0.2, 1.0, 3)
        coloured[segments == sid] = colour
    blended = (alpha * rgb + (1 - alpha) * coloured).clip(0, 1)
    result = mark_boundaries(blended, segments, color=(1, 1, 1), mode="thick")
    return (result * 255).astype(np.uint8)


def iou(pred: np.ndarray, gt: np.ndarray) -> float:
    pred, gt = pred.astype(bool), gt.astype(bool)
    inter = (pred & gt).sum()
    union = (pred | gt).sum()
    return float(inter / union) if union > 0 else float("nan")


print("Helpers loaded.")

In [ ]:
# Image paths
area = "nordvestsmall"
base_dir = Path(f"images/{area}")
poly_path = Path(f"poly/{area}.gpkg")

image_ids = [
    "2023_84_40_2_0138_00062641",
    "2023_84_40_1_0203_00120805",
    "2023_84_40_3_0203_00120820",
]
xs = [2500,  700, 1300]
ys = [5600, 3900, 2500]
tile_size = 2500

token = os.getenv("DATAFORSYNINGEN_TOKEN", "a91f6ab4fc061c7bcc8fc0feea5ef170")

# Load one full image at a time, crop the tile, then delete the full array.
# This avoids holding 3 x ~1 GB arrays in RAM simultaneously.
cams, tiles = [], []

for img_id, x0, y0 in zip(image_ids, xs, ys):
    img = cv2.imread(str(base_dir / f"{img_id}.tif"))
    if img is None:
        raise FileNotFoundError(f"Missing: {base_dir}/{img_id}.tif")
    item = fetch_item(build_item_url("skraafotos" + img_id[:4], img_id, token))
    cam  = build_camera(item)
    cams.append(cam)
    tiles.append(img[y0:y0+tile_size, x0:x0+tile_size].copy())
    print(f"{img_id[-8:]}  full={img.shape[1]}x{img.shape[0]}  "
          f"tile origin=({x0},{y0})  direction={cam.direction}")
    del img     # release immediately
    gc.collect()

gdf = gpd.read_file(poly_path)
print(f"\nBuilding GDF: {len(gdf)} polygons, CRS={gdf.crs}")

show(*tiles,
     titles=[f"Tile {i+1} - {iid[-8:]}" for i, iid in enumerate(image_ids)],
     figsize_per=(6, 6))


In [ ]:
# SAM 2 configuration
SAM_CHECKPOINT = "checkpoints/sam2_hiera_large.pt"
SAM_CONFIG = "sam2_hiera_l"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

_sam2_model = None
_sam2_predictor = None
_sam2_auto_gen = None


def _load_model():
    global _sam2_model
    if _sam2_model is None:
        from sam2.build_sam import build_sam2
        _sam2_model = build_sam2(SAM_CONFIG, SAM_CHECKPOINT, device=device)
        print("SAM2 model loaded.")
    return _sam2_model


def get_sam2_predictor():
    global _sam2_predictor
    if _sam2_predictor is None:
        from sam2.sam2_image_predictor import SAM2ImagePredictor
        _sam2_predictor = SAM2ImagePredictor(_load_model())
    return _sam2_predictor


def get_sam2_auto_generator(**kwargs):
    global _sam2_auto_gen
    from sam2.automatic_mask_generator import SAM2AutomaticMaskGenerator
    defaults = dict(
        points_per_side=32,
        points_per_batch=64,
        pred_iou_thresh=0.82,
        stability_score_thresh=0.85,
        box_nms_thresh=0.7,
        crop_n_layers=1,               # also segment at half resolution
        crop_overlap_ratio=0.5,
        crop_n_points_downscale_factor=2,
        min_mask_region_area=1500,
        output_mode="uncompressed_rle",
    )
    defaults.update(kwargs)
    _sam2_auto_gen = SAM2AutomaticMaskGenerator(_load_model(), **defaults)
    return _sam2_auto_gen


print("SAM2 loaders defined (model not yet loaded).")

def rle_to_mask(rle_dict):
    """Decode SAM uncompressed RLE to a bool mask"""
    counts = rle_dict["counts"]
    h, w = rle_dict["size"]
    flat = np.zeros(h * w, dtype=bool)
    pos, val = 0, False
    for run in counts:
        flat[pos:pos+run] = val
        pos += run
        val  = not val
    return flat.reshape(h, w, order="F")   # SAM uses column-major order


def masks_to_label_image_from_rle(masks_list):
    """
    Build label image from SAM output without materialising all bool arrays at once
    """
    if not masks_list:
        return np.zeros((1, 1), dtype=np.uint32), pd.DataFrame()

    # Infer size from first entry
    first_seg = masks_list[0]["segmentation"]
    if isinstance(first_seg, dict):
        h, w = first_seg["size"]
    else:
        h, w = first_seg.shape

    label_img = np.zeros((h, w), dtype=np.uint32)
    stats_rows = []

    for idx, m in enumerate(sorted(masks_list, key=lambda x: x["area"], reverse=True), 1):
        seg = m["segmentation"]
        if isinstance(seg, dict):
            mask = rle_to_mask(seg)
        else:
            mask = seg.astype(bool)
        label_img[mask] = idx
        ys_s, xs_s = np.where(mask)
        stats_rows.append({"id": idx, "area_px": int(mask.sum()),
                            "cy": float(ys_s.mean()), "cx": float(xs_s.mean())})
        del mask

    return label_img, pd.DataFrame(stats_rows)


def segment_overlay(img_bgr, segments, alpha=0.6):
    """Colour each segment a random hue and show boundaries."""
    rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB).astype(float) / 255
    coloured = np.zeros_like(rgb)
    rng = np.random.default_rng(42)
    for sid in np.unique(segments):
        if sid == 0:
            continue
        coloured[segments == sid] = rng.uniform(0.2, 1.0, 3)
    blended = (alpha * rgb + (1 - alpha) * coloured).clip(0, 1)
    result = mark_boundaries(blended, segments, color=(1, 1, 1), mode="thick")
    return (result * 255).astype(np.uint8)


# Run S3
s3_results = []
gen = get_sam2_auto_generator()

for i, tile in enumerate(tiles):
    tile_rgb = cv2.cvtColor(tile, cv2.COLOR_BGR2RGB)

    with Benchmark() as bm:
        sam_masks = gen.generate(tile_rgb)

    label_img, stats = masks_to_label_image_from_rle(sam_masks)
    del sam_masks
    gc.collect()

    s3_results.append({
        "label_img":  label_img,
        "stats":      stats,
        "n_segments": len(stats),
        "benchmark":  bm,
    })
    print(f"Tile {i+1}: {len(stats)} segments  {bm}")
    print(stats["area_px"].describe().to_string())
    print()

ovs_s3 = [segment_overlay(tile, r["label_img"]) for tile, r in zip(tiles, s3_results)]
show(*ovs_s3,
     figsize_per=(6, 6), bgr=False)


print(f"S3 Auto - Tile {i+1} ({r['n_segments']} segs)" for i, r in enumerate(s3_results))


In [ ]:
# ── DINO configuration ───────────────────────────────────────────────────────
DINO_CONFIG  = "GroundingDINO/groundingdino/config/GroundingDINO_SwinT_OGC.py"
DINO_WEIGHTS = "checkpoints/groundingdino_swint_ogc.pth"

# Urban classes for oblique imagery, ordered loosely by importance
URBAN_CLASSES = [
    "building",
    #"roof",
    "vegetation",
    "grass",
    "road",
    "car",
    "window",
]

# Colour map for the urban class overlay
URBAN_COLOURS = {
    "building":   (255,165,0), # orange  (BGR)
    #"roof":       (0,0,200), # dark red
    "vegetation": (0,180,0), # green
    "grass":      (0,230,100), # lime
    "road":       (0,255,255), # yellow
    "car":        (0,0,255), # red
    "window":     (255,255,0), # cyan
}

def build_text_prompt(classes):
    return " . ".join(classes) + " ."

print("DINO config defined. Prompt:", build_text_prompt(URBAN_CLASSES))

from torchvision import transforms as T
from groundingdino.util.inference import load_model, predict
from groundingdino.util import box_ops

dino_transform = T.Compose([
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

dino_model = load_model(DINO_CONFIG, DINO_WEIGHTS)
dino_model = dino_model.to(device)
print("Grounding DINO loaded.")

In [ ]:

DINO_TILE_SIZE  = 800 # px matches DINO training resolution
DINO_TILE_STEP  = 600 # px 200px overlap between adjacent sub-tiles


def _nms_boxes(boxes, scores, iou_thresh=0.5):
    """Pure-numpy NMS. boxes: (N,4) xyxy, scores: (N,). Returns kept indices."""
    if len(boxes) == 0:
        return np.array([], dtype=int)
    order = np.argsort(scores)[::-1]
    keep  = []
    while len(order):
        i = order[0]
        keep.append(i)
        if len(order) == 1:
            break
        xx1 = np.maximum(boxes[i, 0], boxes[order[1:], 0])
        yy1 = np.maximum(boxes[i, 1], boxes[order[1:], 1])
        xx2 = np.minimum(boxes[i, 2], boxes[order[1:], 2])
        yy2 = np.minimum(boxes[i, 3], boxes[order[1:], 3])
        w   = np.maximum(0.0, xx2 - xx1)
        h   = np.maximum(0.0, yy2 - yy1)
        inter = w * h
        area_i = (boxes[i, 2] - boxes[i, 0]) * (boxes[i, 3] - boxes[i, 1])
        area_j = ((boxes[order[1:], 2] - boxes[order[1:], 0]) *
                  (boxes[order[1:], 3] - boxes[order[1:], 1]))
        iou    = inter / (area_i + area_j - inter + 1e-6)
        order  = order[1:][iou < iou_thresh]
    return np.array(keep, dtype=int)


def run_dino_on_subtile(subtile_rgb, box_threshold=0.28, text_threshold=0.23):
    """Run DINO on a single <=800px RGB crop. return normalised-cx/cy/w/h boxes."""
    from groundingdino.util import box_ops as _bops
    tensor = dino_transform(subtile_rgb)
    boxes_norm, logits, phrases = predict(
        model=dino_model,
        image=tensor,
        caption=build_text_prompt(URBAN_CLASSES),
        box_threshold=box_threshold,
        text_threshold=text_threshold,
        device=device,
    )
    if len(boxes_norm) == 0:
        return np.empty((0, 4)), np.empty(0), []
    boxes_xyxy = _bops.box_cxcywh_to_xyxy(boxes_norm).cpu().numpy()
    logits_np  = logits.cpu().numpy()
    return boxes_xyxy, logits_np, phrases   # normalised [0,1] xyxy


def run_dino_tiled(tile_bgr,
                   sub_size=DINO_TILE_SIZE,
                   step=DINO_TILE_STEP,
                   box_threshold=0.28,
                   text_threshold=0.23,
                   global_nms_iou=0.45):
    H, W   = tile_bgr.shape[:2]
    all_boxes, all_scores, all_phrases = [], [], []

    ys_starts = list(range(0, H - sub_size + 1, step))
    xs_starts = list(range(0, W - sub_size + 1, step))
    # Make sure we always cover the bottom/right edge
    if not ys_starts or ys_starts[-1] + sub_size < H:
        ys_starts.append(max(0, H - sub_size))
    if not xs_starts or xs_starts[-1] + sub_size < W:
        xs_starts.append(max(0, W - sub_size))

    n_crops = len(ys_starts) * len(xs_starts)
    print(f"  DINO: {n_crops} sub-tiles ({sub_size}px, step={step}px)")

    for y0 in ys_starts:
        for x0 in xs_starts:
            crop_bgr = tile_bgr[y0:y0+sub_size, x0:x0+sub_size]
            crop_rgb = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB)
            sh, sw   = crop_bgr.shape[:2]

            boxes_norm, scores, phrases = run_dino_on_subtile(
                crop_rgb, box_threshold, text_threshold
            )
            if len(boxes_norm) == 0:
                continue

            # Remap from normalised [0,1] to tile pixel coords
            boxes_px       = boxes_norm.copy()
            boxes_px[:, 0] = boxes_norm[:, 0] * sw + x0 # x1
            boxes_px[:, 1] = boxes_norm[:, 1] * sh + y0 # y1
            boxes_px[:, 2] = boxes_norm[:, 2] * sw + x0 # x2
            boxes_px[:, 3] = boxes_norm[:, 3] * sh + y0 # y2

            all_boxes.append(boxes_px)
            all_scores.append(scores)
            all_phrases.extend(phrases)

    if not all_boxes:
        return np.empty((0, 4)), []

    boxes_cat  = np.vstack(all_boxes)
    scores_cat = np.concatenate(all_scores)

    # Global NMS across all sub-tile detections
    keep = _nms_boxes(boxes_cat, scores_cat, iou_thresh=global_nms_iou)
    phrases_kept = [all_phrases[k] for k in keep]
    print(f"  DINO: {len(boxes_cat)} raw -> {len(keep)} after NMS")

    return boxes_cat[keep], phrases_kept


def phrase_to_class(phrase):
    for cls in URBAN_CLASSES:
        if cls in phrase.lower():
            return cls
    return "other"


def run_sam_boxes_to_class_masks(tile_bgr, boxes_xyxy, phrases, tile_hw):
    """
    Run SAM2 on boxes in small batches, reduce each mask to per-class bool
    union on-the-fly. Never accumulates the full (N, H, W) stack.
    """
    predictor = get_sam2_predictor()
    tile_rgb  = cv2.cvtColor(tile_bgr, cv2.COLOR_BGR2RGB)
    predictor.set_image(tile_rgb)

    H, W        = tile_hw
    class_masks = {cls: np.zeros((H, W), dtype=bool) for cls in URBAN_CLASSES}
    label_img   = np.zeros((H, W), dtype=np.uint32)
    scores_out  = []
    det_id      = 1
    BATCH       = 32

    for b0 in range(0, len(boxes_xyxy), BATCH):
        b_boxes   = boxes_xyxy[b0:b0+BATCH]
        b_phrases = phrases[b0:b0+BATCH]

        masks_b, scores_b, _ = predictor.predict(
            point_coords=None,
            point_labels=None,
            box=b_boxes,
            multimask_output=False,
        )
        if masks_b.ndim == 4:
            masks_b = masks_b[:, 0]

        for mask_np, phrase, score in zip(masks_b, b_phrases, scores_b):
            cls    = phrase_to_class(phrase)
            binary = mask_np.astype(bool)
            if cls in class_masks:
                class_masks[cls] |= binary
            label_img[binary] = det_id
            scores_out.append(float(score))
            det_id += 1
            del binary, mask_np

        del masks_b
        gc.collect()

    return class_masks, label_img, scores_out


print("DINO tiled inference + SAM2 helpers defined.")
print(f"Sub-tile grid for {tile_size}px tile: "
      f"{len(range(0, tile_size-DINO_TILE_SIZE+1, DINO_TILE_STEP))+1} x "
      f"{len(range(0, tile_size-DINO_TILE_SIZE+1, DINO_TILE_STEP))+1} crops")


In [ ]:
s4_results = []

for i, tile in enumerate(tiles):
    with Benchmark() as bm:
        # Tiled DINO: slides an 800px window with 200px overlap
        boxes_xyxy, phrases = run_dino_tiled(tile)

        if len(boxes_xyxy) == 0:
            print(f"Tile {i+1}: DINO found no detections after tiling. "
                  "Try lowering box_threshold in run_dino_tiled().")
            s4_results.append(None)
            continue

        class_masks, label_img, scores = run_sam_boxes_to_class_masks(
            tile, boxes_xyxy, phrases, tile.shape[:2]
        )

    detection_summary = {cls: int(class_masks[cls].sum()) for cls in URBAN_CLASSES}

    s4_results.append({
        "boxes":        boxes_xyxy,
        "phrases":      phrases,
        "scores":       scores,
        "class_masks":  class_masks,
        "label_img":    label_img,
        "n_detections": len(phrases),
        "benchmark":    bm,
    })

    print(f"Tile {i+1}: {len(phrases)} detections  {bm}")
    for cls, px in detection_summary.items():
        pct = 100 * px / (tile_size**2)
        if pct > 0.1:
            print(f"  {cls:12s}: {px:>8,} px  ({pct:.1f}%)")

# Visualise
fig, axes = plt.subplots(1, len(tiles), figsize=(18, 6))

for ax, r, tile in zip(axes, s4_results, tiles):
    if r is None:
        ax.axis("off")
        continue

    ov = overlay_masks(tile, r["class_masks"],
                       colours=URBAN_COLOURS, alpha=0.5)
    ov_rgb = cv2.cvtColor(ov, cv2.COLOR_BGR2RGB)

    ax.imshow(ov_rgb)
    ax.axis("off")

plt.tight_layout()
plt.show()
